[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/43_masked_pairwise_dist_solution.ipynb)

# 🟡 Solution: Masked Pairwise Distance

**Primitive: broadcasting (outer-product subtraction) + `masked_fill`**

**Reduction:** `D[i,j]` depends on two independent things computed via outer products:
1. **Distance:** `(points[i] - points[j])` — broadcasting `points[:, None, :]` against `points[None, :, :]`
2. **Mask:** `group_ids[i] == group_ids[j]` — broadcasting `group_ids[:, None]` against `group_ids[None, :]`

This is the same pattern as masked attention — once you see it, it's everywhere.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def masked_pairwise_dist(points: torch.Tensor, group_ids: torch.Tensor) -> torch.Tensor:
    # primitive: outer-product broadcasting for both distance and group mask
    diff = points[:, None, :] - points[None, :, :]  # (N, N, D) — all pairwise differences
    dist = diff.pow(2).sum(-1)                       # (N, N) — squared L2 distances
    same = group_ids[:, None] == group_ids[None, :]  # (N, N) — outer-product equality mask
    return dist.masked_fill(~same, float('inf'))

In [ ]:
# Verify
points    = torch.tensor([[0.,0.],[1.,0.],[0.,1.],[10.,10.]])
group_ids = torch.tensor([0, 0, 1, 0])
D = masked_pairwise_dist(points, group_ids)
print('Output:')
print(D)
print('Diagonal (should be all 0):', D.diagonal().tolist())
print('Symmetric:', torch.allclose(D, D.T, equal_nan=False))

In [ ]:
# Run judge
from torch_judge import check
check("masked_pairwise_dist")